# Generic - Python

All 5 Python examples from [docs/generic.md](https://platob.github.io/yggdryl/generic/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

## Shared vocabulary

In [ ]:
from yggdryl import enums

assert "int64" in enums.DATA_TYPE_IDS
assert enums.IO_MODES == ("overwrite", "append", "merge", "readonly", "random")

In [ ]:
from yggdryl import Scalar

value = Scalar.from_enum("io_mode", "append")
assert (value.enum_kind, value.enum_value, value.enum_ordinal) == ("io_mode", "append", 1)
assert value.as_py() == "append"

## RecordOptions: every encoding's settings

In [ ]:
import pyarrow as pa

from yggdryl import RecordOptions

schema = pa.schema([pa.field("id", pa.int64(), nullable=False)])

# The media type names the encoding, so there is no format argument.
options = RecordOptions("trades.parquet")
options.field = schema
options.batch_row_size = 1024
options.commit_row_size = 10_000

assert str(options.mime_type) == "application/vnd.apache.parquet"
assert options.name == "row"
assert [child.name for child in options.dtype] == ["id"]
assert options.metadata == {}
assert options.field is not None
assert options.batch_row_size == 1024
assert options.commit_row_size == 10_000

# A setting one encoding has reads as None on an encoding that has none.
assert options.max_row_group_size == 1_048_576
assert RecordOptions("trades.arrows").max_row_group_size is None

In [ ]:
from yggdryl import DataType, Field, RecordOptions

schema = Field("row", DataType.from_fields([Field("id", "int64", nullable=False)]), nullable=False)
options = RecordOptions("trades.arrows")
options.field = schema

# One stored form: declaring only the datatype is the same declaration.
by_dtype = RecordOptions("trades.arrows")
by_dtype.dtype = schema.dtype
assert options == by_dtype
assert options.stable_hash() == by_dtype.stable_hash()

options.name = "trade"
assert options.field.name == "trade"
assert options.field.dtype == schema.dtype

options.metadata = {"source": "exchange"}
assert options.field.metadata["source"] == "exchange"

# The setter takes a datatype expression as readily as a DataType.
options.dtype = "struct<id: int64, venue: utf8>"
built = options.field
assert built.name == "trade"
assert [child.name for child in built.dtype] == ["id", "venue"]
assert built.metadata["source"] == "exchange"
assert not built.nullable

# None clears a part; the name stays.
options.dtype = None
options.metadata = None
assert options.field is None
assert options.metadata == {}
assert options.name == "trade"

## TypedScalar: one value and its datatype

In [ ]:
from dataclasses import dataclass

from yggdryl import Scalar

@dataclass
class Row:
    id: int

assert Scalar.from_py(42).into_field().name == "value"
assert Scalar.from_py([1, None]).into_array_field().name == "item"
assert Scalar.from_py([Row(1)]).into_struct_field().name == "row"